# Get data from Google API and convert it into Excel and CSV format

In [1]:
# Import modules

from google.analytics.data_v1beta import BetaAnalyticsDataClient
from google.analytics.data_v1beta.types import (
    DateRange,
    Dimension,
    Metric,
    RunReportRequest,
)
from google.oauth2 import service_account
from google.protobuf.json_format import MessageToDict
import pandas as pd
import os


## Global Variables, property ID, creds

In [5]:
## Set up global variables

os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = 'credentials.json'
property_id = '451093576'

client = BetaAnalyticsDataClient()

## Making first request for Geographic details

In [6]:
# Get the data from the API

request = RunReportRequest(
    property=f"properties/{property_id}",
    dimensions=[
        Dimension(name="audienceId"),
        Dimension(name="country"),
        Dimension(name="region"),
        Dimension(name="city"),
        Dimension(name="date"),
        Dimension(name="newVsReturning"),
        Dimension(name="platform")
    ],
    metrics=[
        Metric(name="averageSessionDuration"),
        Metric(name="eventCount"),
        Metric(name="totalUsers")
    ],
    date_ranges=[DateRange(start_date="2020-03-31", end_date="today")],
    )

response = client.run_report(request)

## Making second request for Tech details

In [7]:
# Get the data from the API

request2 = RunReportRequest(
    property=f"properties/{property_id}",
    dimensions=[
        Dimension(name="audienceId"),
        Dimension(name="browser"),
        Dimension(name="deviceCategory"),
        Dimension(name="deviceModel"),
        Dimension(name="pageTitle"),
        Dimension(name="newVsReturning"),
        Dimension(name="operatingSystem"),
        Dimension(name="operatingSystemVersion"),
        Dimension(name="sessionSource")
    ],
    metrics=[
        Metric(name="activeUsers"),
        Metric(name="sessions")],
    date_ranges=[DateRange(start_date="2020-03-31", end_date="today")],
    )

response2 = client.run_report(request2)

## Making third request for Event names and their count

In [8]:
# Get the data from the API

request3 = RunReportRequest(
    property=f"properties/{property_id}",
    dimensions=[
        Dimension(name="audienceId"),
        Dimension(name="eventName"),
        Dimension(name="platform")
    ],
    metrics=[
        Metric(name="eventCount")
    ],
    date_ranges=[DateRange(start_date="2024-01-01", end_date="today")],
    )

response3 = client.run_report(request3)

## Function for converting the received data into a formatted table

In [9]:
# Turn the raw data into a Table

def ga4_result_to_df(response):
    """Original: print_run_report_response: Prints results of a runReport call. v2.1 changed by Bram to create DataFrame"""
    result_dict = {}
    for dimensionHeader in response.dimension_headers:
        result_dict[dimensionHeader.name] = []
    for metricHeader in response.metric_headers:
        result_dict[metricHeader.name] = []
    for rowIdx, row in enumerate(response.rows):
        for i, dimension_value in enumerate(row.dimension_values):
            dimension_name = response.dimension_headers[i].name
            result_dict[dimension_name].append(dimension_value.value)
        for i, metric_value in enumerate(row.metric_values):
            metric_name = response.metric_headers[i].name
            result_dict[metric_name].append(metric_value.value)
    return pd.DataFrame(result_dict)



## Location Details

In [10]:
df = ga4_result_to_df(response)
df

,audienceId,country,region,city,date,newVsReturning,platform,averageSessionDuration,eventCount,totalUsers
0,8495975418,India,Maharashtra,Pune,20240810,returning,web,2048.3265100999997,131,5
1,8495975418,India,Kerala,(not set),20240812,new,web,1558.5602645,35,2
2,8495975418,Germany,Baden-Wurttemberg,Heidelberg,20240802,returning,web,1379.3460495,12,2
3,8495975418,Germany,Hessen,Frankfurt,20240812,new,web,1320.867469,12,2
4,8495975418,Germany,North Rhine-Westphalia,(not set),20240812,new,web,1319.0179885,13,2
...,...,...,...,...,...,...,...,...,...,...
725,8495975418,United States,Ohio,Columbus,20240816,new,web,0,3,1
726,8495975418,United States,Oregon,Boardman,20240726,new,web,0,3,1
727,8495975418,United States,Oregon,Boardman,20240819,new,web,0,3,1
728,8495975418,United States,Tennessee,Farragut,20240723,new,web,0,3,1


## Formatting the output (Date and Time)

In [11]:

# Convert the sessionDuration column to floats and then to integers
df['averageSessionDuration'] = df['averageSessionDuration'].astype(float).astype(int)


# Function to convert seconds to HH:MM format
def convert_seconds_to_hhmm(seconds):
    hours = seconds // 3600
    minutes = (seconds % 3600) // 60
    return f'{hours:02}:{minutes:02}'

# Apply the function to the sessionDuration column
df['avgSessionDuration_HHMM'] = df['averageSessionDuration'].apply(convert_seconds_to_hhmm)

# Drop the original sessionDuration column
df.drop('averageSessionDuration', axis=1, inplace=True)




# Convert the date column to datetime format
df['date'] = pd.to_datetime(df['date'], format='%Y%m%d')

# Reformat the date to DD-MM-YYYY
df['formatted_date'] = df['date'].dt.strftime('%d-%m-%Y')

df.drop('date', axis=1, inplace=True)

df

,audienceId,country,region,city,newVsReturning,platform,eventCount,totalUsers,avgSessionDuration_HHMM,formatted_date
0,8495975418,India,Maharashtra,Pune,returning,web,131,5,00:34,10-08-2024
1,8495975418,India,Kerala,(not set),new,web,35,2,00:25,12-08-2024
2,8495975418,Germany,Baden-Wurttemberg,Heidelberg,returning,web,12,2,00:22,02-08-2024
3,8495975418,Germany,Hessen,Frankfurt,new,web,12,2,00:22,12-08-2024
4,8495975418,Germany,North Rhine-Westphalia,(not set),new,web,13,2,00:21,12-08-2024
...,...,...,...,...,...,...,...,...,...,...
725,8495975418,United States,Ohio,Columbus,new,web,3,1,00:00,16-08-2024
726,8495975418,United States,Oregon,Boardman,new,web,3,1,00:00,26-07-2024
727,8495975418,United States,Oregon,Boardman,new,web,3,1,00:00,19-08-2024
728,8495975418,United States,Tennessee,Farragut,new,web,3,1,00:00,23-07-2024


## Tech Details

In [12]:
df2 = ga4_result_to_df(response2)
df2

,audienceId,browser,deviceCategory,deviceModel,pageTitle,newVsReturning,operatingSystem,operatingSystemVersion,sessionSource,activeUsers,sessions
0,8495975418,Android Webview,mobile,(not set),Home - Egenie4u,new,Android,14,l.instagram.com,198,434
1,8495975418,Safari (in-app),mobile,(not set),Home - Egenie4u,new,iOS,17.5.1,l.instagram.com,179,605
2,8495975418,Android Webview,mobile,(not set),Laptops - Egenie Stores,new,Android,14,l.instagram.com,120,256
3,8495975418,Safari (in-app),mobile,(not set),Consoles Archives - Egenie Stores,new,iOS,17.5.1,l.instagram.com,106,367
4,8495975418,Android Webview,mobile,(not set),Home - Egenie4u,new,Android,13,l.instagram.com,76,167
...,...,...,...,...,...,...,...,...,...,...,...
2270,8495975418,Android Webview,mobile,(not set),Home - Egenie4u,(not set),Android,14,l.instagram.com,0,1
2271,8495975418,Chrome,desktop,(not set),Contact Us - Egenie Stores,(not set),Windows,11,(direct),0,1
2272,8495975418,Chrome,desktop,(not set),Home - Egenie4u,(not set),Windows,10,(direct),0,2
2273,8495975418,Chrome,desktop,(not set),Home - Egenie4u,(not set),Windows,11,tagassistant.google.com,0,1


## Event Details

In [13]:
df3 = ga4_result_to_df(response3)
df3

,audienceId,eventName,platform,eventCount
0,8495975418,page_view,web,9158
1,8495975418,user_engagement,web,6475
2,8495975418,session_start,web,1917
3,8495975418,first_visit,web,1735
4,8495975418,scroll,web,844
5,8495975418,form_start,web,333
6,8495975418,form_submit,web,321
7,8495975418,view_search_results,web,209
8,8495975418,click,web,8


## Merging data into one table

In [14]:
# Merge DataFrames on audienceId and newVsReturning columns
merged_df = pd.merge(df, df2, on=['audienceId', 'newVsReturning'], how='inner')
merged_df

,audienceId,country,region,city,newVsReturning,platform,eventCount,totalUsers,avgSessionDuration_HHMM,formatted_date,browser,deviceCategory,deviceModel,pageTitle,operatingSystem,operatingSystemVersion,sessionSource,activeUsers,sessions
0,8495975418,India,Maharashtra,Pune,returning,web,131,5,00:34,10-08-2024,Android Webview,mobile,(not set),Home - Egenie4u,Android,14,l.instagram.com,5,10
1,8495975418,India,Maharashtra,Pune,returning,web,131,5,00:34,10-08-2024,Android Webview,mobile,(not set),Gaming Laptops Archives - Egenie Stores,Android,11,l.instagram.com,3,4
2,8495975418,India,Maharashtra,Pune,returning,web,131,5,00:34,10-08-2024,Android Webview,mobile,(not set),Home - Egenie4u,Android,11,l.instagram.com,3,4
3,8495975418,India,Maharashtra,Pune,returning,web,131,5,00:34,10-08-2024,Android Webview,mobile,(not set),Laptops - Egenie Stores,Android,11,l.instagram.com,3,4
4,8495975418,India,Maharashtra,Pune,returning,web,131,5,00:34,10-08-2024,Chrome,desktop,(not set),Home - Egenie4u,Windows,10,(direct),3,7
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1289957,8495975418,United States,Washington,Seattle,new,web,3,1,00:00,21-08-2024,Samsung Internet,mobile,(not set),Lenovo LOQ 2024 Intel i5-13450HX - Egenie Stores,Android,10,l.instagram.com,1,1
1289958,8495975418,United States,Washington,Seattle,new,web,3,1,00:00,21-08-2024,Samsung Internet,mobile,(not set),"Lenovo LOQ Intel Core i5-12450HX 15.6"" (39.6cm...",Android,10,(direct),1,1
1289959,8495975418,United States,Washington,Seattle,new,web,3,1,00:00,21-08-2024,Samsung Internet,mobile,(not set),"Lenovo LOQ Intel Core i5-12450HX 15.6"" (39.6cm...",Android,10,l.instagram.com,1,1
1289960,8495975418,United States,Washington,Seattle,new,web,3,1,00:00,21-08-2024,Samsung Internet,mobile,SM-S711B,Lenovo LOQ 2024 Intel i5-13450HX - Egenie Stores,Android,10,(direct),1,1


In [15]:
# Rename columns
new_column_names = {
    'audienceId': 'Audience ID',
    'country': 'Country',
    'region': 'Region',
    'city': 'City',
    'newVsReturning': 'New/Returning',
    'platform':'Platform',
    'eventCount': 'Event Count',
    'totalUsers': 'Total Users',
    'avgSessionDuration_HHMM': 'Avg Session Duration',
    'formatted_date': 'Date',
    'browser': 'Browser',
    'deviceCategory': 'Device Category',
    'deviceModel': 'Device Model',
    'pageTitle': 'Page Title',
    'operatingSystem': 'OS',
    'operatingSystemVersion': 'OS Version',
    'sessionSource': 'Session Source',
    'activeUsers': 'Active Users',
    'sessions': 'Sessions',
}
merged_df.rename(columns=new_column_names, inplace=True)
merged_df

,Audience ID,Country,Region,City,New/Returning,Platform,Event Count,Total Users,Avg Session Duration,Date,Browser,Device Category,Device Model,Page Title,OS,OS Version,Session Source,Active Users,Sessions
0,8495975418,India,Maharashtra,Pune,returning,web,131,5,00:34,10-08-2024,Android Webview,mobile,(not set),Home - Egenie4u,Android,14,l.instagram.com,5,10
1,8495975418,India,Maharashtra,Pune,returning,web,131,5,00:34,10-08-2024,Android Webview,mobile,(not set),Gaming Laptops Archives - Egenie Stores,Android,11,l.instagram.com,3,4
2,8495975418,India,Maharashtra,Pune,returning,web,131,5,00:34,10-08-2024,Android Webview,mobile,(not set),Home - Egenie4u,Android,11,l.instagram.com,3,4
3,8495975418,India,Maharashtra,Pune,returning,web,131,5,00:34,10-08-2024,Android Webview,mobile,(not set),Laptops - Egenie Stores,Android,11,l.instagram.com,3,4
4,8495975418,India,Maharashtra,Pune,returning,web,131,5,00:34,10-08-2024,Chrome,desktop,(not set),Home - Egenie4u,Windows,10,(direct),3,7
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1289957,8495975418,United States,Washington,Seattle,new,web,3,1,00:00,21-08-2024,Samsung Internet,mobile,(not set),Lenovo LOQ 2024 Intel i5-13450HX - Egenie Stores,Android,10,l.instagram.com,1,1
1289958,8495975418,United States,Washington,Seattle,new,web,3,1,00:00,21-08-2024,Samsung Internet,mobile,(not set),"Lenovo LOQ Intel Core i5-12450HX 15.6"" (39.6cm...",Android,10,(direct),1,1
1289959,8495975418,United States,Washington,Seattle,new,web,3,1,00:00,21-08-2024,Samsung Internet,mobile,(not set),"Lenovo LOQ Intel Core i5-12450HX 15.6"" (39.6cm...",Android,10,l.instagram.com,1,1
1289960,8495975418,United States,Washington,Seattle,new,web,3,1,00:00,21-08-2024,Samsung Internet,mobile,SM-S711B,Lenovo LOQ 2024 Intel i5-13450HX - Egenie Stores,Android,10,(direct),1,1


In [16]:
merged_df.to_csv('GA4_python_output.csv')
df3.to_csv('Event_details.csv')

In [17]:

## Export to Excel
# Export to Excel without the index column
merged_df.to_excel('GA4_python_output.xlsx', sheet_name='GA4_report', index=False, engine='xlsxwriter')
df3.to_excel('Event details.xlsx', sheet_name='Event report', index=False, engine='xlsxwriter')


ValueError: This sheet is too large! Your sheet size is: 1289962, 19 Max sheet size is: 1048576, 16384